# Preparation Dinerstein et al. 2017 data

In [ ]:
# libraries
import os, time, shutil
import numpy as np
import xarray as xr
import rioxarray
import geopandas as gpd
import rasterio
from rasterio import features
from  scipy import ndimage
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir_data = '../data/'
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'

---

### Pre-processing

In [ ]:
# Dinerstein et al. categories
# 1	 Tropical & Subtropical Moist Broadleaf Forests
# 2	 Tropical & Subtropical Dry Broadleaf Forests
# 3	 Tropical & Subtropical Coniferous Forests
# 4	 Temperate Broadleaf & Mixed Forests
# 5	 Temperate Conifer Forests
# 6	 Boreal Forests/Taiga
# 7	 Tropical & Subtropical Grasslands, Savannas & ...
# 8	 Temperate Grasslands, Savannas & Shrublands
# 9	 Flooded Grasslands & Savannas
# 10 Montane Grasslands & Shrublands
# 11 Tundra
# 12 Mediterranean Forests, Woodlands & Scrub
# 13 Deserts & Xeric Shrublands
# 14 Mangroves

In [ ]:
def prep_dinerstein_tif():
    
    """Prepare Dinerstein shapefile and export as tif"""
    
    # Get target tif
    rst = rasterio.open(f'{dir_data}worldclim/wc2.1_30s_elev/wc2.1_30s_elev.tif')
    
    # Get shapefile
    shp0 = gpd.read_file(f'{dir_data}dinerstein2017/Ecoregions2017.shp')

    # Get biome names
    shp_name = shp0[['BIOME_NUM', 'BIOME_NAME']] \
        .rename(columns = {'BIOME_NUM': 'biome_num', 'BIOME_NAME': 'biome_name'}) \
        .drop_duplicates() \
        .sort_values('biome_num')
    
    shp_name = shp_name[shp_name['biome_name'] != 'N/A'] \
        .reset_index(drop=True)
    
    # Get relevant shapefile information
    # Drop polygon with id 207 "Rock and Ice" (misclassified as tundra)
    shp = shp0[shp0['OBJECTID'] != 207][['BIOME_NUM', 'geometry']] \
        .dissolve(by='BIOME_NUM', aggfunc='sum') \
        .reset_index() \
        .rename(columns = {'BIOME_NUM': 'biome_num'})
      
    # Reproject
    shp = shp.to_crs("EPSG:4326")
    
    # Get meta data from target raster
    meta = rst.meta.copy()
    meta.update(compress='lzw')
    
    # Export as raster
    def gpd_to_tif(shp, out_fn):
        with rasterio.open(out_fn, 'w+', **meta) as out:
            out_arr = out.read(1)
            
            shapes = ((geom,value) for geom, value in 
                      zip(shp.geometry, shp['biome_num']))
    
            burned = features.rasterize(shapes=shapes, 
                                        fill=0, 
                                        out=out_arr, 
                                        transform=out.transform)
    
            out.write_band(1, burned)
    
    
    gpd_to_tif(shp, f'{dir01}dinerstein_biomes.tif')
    
    # Get tif
    da_biomes = rioxarray.open_rasterio(f'{dir01}dinerstein_biomes.tif', 
                                        chunks=dict(y=5000, x=5000))
    
    # Mask non-land
    da_biomes = da_biomes \
        .where(da_biomes != -32768)[0] \
        .drop_vars(['spatial_ref', 'band'])
    # Select lat between 80 and -60
    da_biomes = da_biomes.sel(y=slice(80, -60))    
    # Add attributes
    da_biomes.attrs['long_name'] = 'dinerstein_biomes'
    # Export
    da_biomes.rio.to_raster(f'{dir01}dinerstein_biomes.tif')


%time prep_dinerstein_tif()

---

### Fill land nans 0.01°

In [ ]:
def fill_nans(var, dir_out):
    """
    Fills NaN values in the specified variable's dataset using the nearest valid 
    data, applies a land mask, and exports the result to a new Zarr dataset.

    Args:
        var (str): Name of the variable to process (e.g., 'temperature', 'precipitation').
        dir_out (str): Directory where prepared data is stored and the filled dataset will be exported.

    Returns:
        None
    """

    def fill_nans_array(data, invalid):
        """
        Replace invalid (NaN) data cells by the value of the nearest valid data 
        cell.
        """
        ind = ndimage.distance_transform_edt(invalid,
                                             return_distances=False,
                                             return_indices=True)
        return data[tuple(ind)]

    # Paths for input and output
    land_mask_path = os.path.join(dir_out, 'ds_prep_copernicus_land_mask_r001.zarr')
    var_data_path = os.path.join(dir_out, f'ds_regridded_{var}.zarr')
    output_path = os.path.join(dir_out, f'ds_prep_{var}_r001.zarr')

    # Read land mask data
    da_land = xr.open_zarr(land_mask_path) \
                .chunk(dict(lat=5000, lon=5000)) \
                .copernicus_land_mask \
                .compute()

    # Read variable data
    da_var = rioxarray.open_rasterio(f'{dir01}{var}.tif', 
                                     chunks=dict(y=5000, x=5000))[0] \
        .rename(y='lat', x='lon') \
        .sel(lat=slice(da_land.lat.max(), da_land.lat.min()),
             lon=slice(da_land.lon.min(), da_land.lon.max()))

    # Fill nan using function fill_nans_array
    # If there are no NaNs, skip filling process
    if not da_var.isnull().any():
        da_fill = da_var.values  # No filling required
    else:
        da_fill = fill_nans_array(da_var.values, da_var.isnull().values)

    # Create a new Dataset with filled data
    ds_filled = xr.Dataset(dict(lat = da_var.lat, lon=da_var.lon))
    ds_filled[var] = (('lat', 'lon'), da_fill)

    # Apply land mask to the filled data
    ds_filled = ds_filled.where(da_land)

    # Export the filled dataset to Zarr format
    ds_filled \
        .drop_vars(['band', 'spatial_ref']) \
        .chunk(dict(lat=5000, lon=5000)) \
        .to_zarr(output_path, mode='w')


%time fill_nans('dinerstein_biomes', dir01)

---

### Regridd 0.01° data to 0.25° (to get fractional values at 0.25°)

In [ ]:
def prep_dinerstein_for_regridding_to_r025():

    # Get prepared dinerstein data
    da = xr.open_zarr(f'{dir01}ds_prep_dinerstein_biomes_r001.zarr') \
        .dinerstein_biomes
    
    # Create dataset for output
    ds_out = xr.Dataset()
    # Add variable for each category
    for i in range(1, 15):
        ds_out['cat' + str(i)] = xr.where(da == i, 1, 0).astype('float32')
    
    # Attributes of latitude and longitude
    ds_out.lat.attrs = dict(standard_name = 'latitude',
                        long_name = 'latitude',
                        units = 'degrees_north',
                        axis = 'Y')
    ds_out.lon.attrs = dict(standard_name = 'longitude',
                        long_name = 'longitude',
                        units = 'degrees_east',
                        axis = 'X')
    
    # Export each variable to seperate dataset (required for regridding)
    for i in range(1, 15): 
        ds_out['cat' + str(i)] \
            .compute() \
            .to_netcdf(f'{dir01}ds_dinerstein_interm_cat{str(i)}.nc', 
                       mode='w')


prep_dinerstein_for_regridding_to_r025()

In [ ]:
%%bash 
cd ../paper_ecosystem_carbon/output/01_data_prep/

module load cdo

for i in cat1 cat2 cat3 cat4 cat5 cat6 cat7 cat8 cat9 cat10 cat11 cat12 cat13 \
    cat14
do
    cdo -P 24 remapycon,../data/luh2_v2h/states.nc \
    ds_dinerstein_interm_${i}.nc ds_dinerstein_interm_r025_${i}.nc
    
done

In [ ]:
# Create one zarr dataset with r025 files
xr.open_mfdataset(f'{dir01}ds_dinerstein_interm_r025_*.nc') \
    .to_zarr(f'{dir01}ds_prep_dinerstein_biomes_r025.zarr', mode='w');

In [ ]:
# Remove intermediate files
for i in range(1, 15):
    os.remove(f'{dir01}ds_dinerstein_interm_cat{str(i)}.nc')
    os.remove(f'{dir01}ds_dinerstein_interm_r025_cat{i}.nc')

---

### Post-processing 0.25°

In [ ]:
# Get regridded dinerstein data
ds_din = xr.open_zarr(dir01 + 'ds_prep_dinerstein_biomes_r025.zarr')

# prep dinerstein data
l_din_trop = [1, 2, 3, 7, 14]
l_din_temp = [4, 5, 6, 8, 12]
l_din_shrub = [1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14]
l_din_grass = [1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 13, 14]
l_din_tundr = [11]
l_din_mount = [10]


def get_da_din(l):
    return xr.where(ds_din[['cat' + str(i) for i in l]] \
                    .to_dataarray('cat').sum('cat') > 0, True, False)
    
ds_din_etype = xr.Dataset()
ds_din_etype['trop_ef'] = get_da_din(l_din_trop)
ds_din_etype['trop_df'] = get_da_din(l_din_trop)
ds_din_etype['temp_ebf'] = get_da_din(l_din_temp)
ds_din_etype['temp_dbf'] = get_da_din(l_din_temp)
ds_din_etype['temp_enf'] = get_da_din(l_din_temp)
ds_din_etype['temp_dnf'] = get_da_din(l_din_temp)
ds_din_etype['glob_shrub'] = get_da_din(l_din_shrub)
ds_din_etype['glob_grass'] = get_da_din(l_din_grass)
ds_din_etype['glob_tundr'] = get_da_din(l_din_tundr)
ds_din_etype['glob_mount'] = get_da_din(l_din_mount)


ds_din_etype.to_zarr(dir01 + 'ds_prep_dinerstein_etype_r025.zarr', mode='w')